In [1]:
from shutil import copy
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from glob import glob
import psi4
import sys
sys.path.insert(0, '/home/amousso3/DDLUCJ/check_amplitudes/')
from helper_CC_ML_spacial import *


  Threads set to 12 by Python driver.


In [9]:
# === 1. Molecule Setup ===
with open('/home/amousso3/DDLUCJ/check_amplitudes/diatomics/NN.xyz','r') as f:
    text=f.read()

qmol = psi4.qcdb.Molecule.from_string(text, dtype='xyz')
mol = psi4.geometry(qmol.create_psi4_string_from_molecule()+ 'symmetry c1')   


psi4.core.clean()
psi4.core.be_quiet()

# === 2. Set Options and Run RHF ===

psi4.set_options({'basis': 'STO-3G',
                  'scf_type':     'pk',
                  'reference':    'rhf',
                  'mp2_type':     'conv',
                  'e_convergence': 1e-8,
                  'd_convergence': 1e-8})

rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)

# === 3. Use HelperCCEnergy ===
A = HelperCCEnergy(mol, rhf_e, scf_wfn, freeze_core=True)
A.compute_energy()

# === 4. Define Active Space ===
n_frozen = A.nfzc  # automatically determined by HelperCCEnergy if freeze_core=True
nmo = A.nmo
active_orbitals = list(range(n_frozen, nmo))

num_orbitals = len(active_orbitals)
n_elec_total = A.ndocc * 2
active_elec = n_elec_total - 2 * n_frozen
num_elec_a = int((active_elec + mol.multiplicity() - 1) // 2)
num_elec_b = int((active_elec - mol.multiplicity() + 1) // 2)

# === 5. Extract Active-Space Integrals ===
hcore = A.H1[np.ix_(active_orbitals, active_orbitals)]

# Get full MO ERI tensor and slice to active space
# No slicing — this is already the active-space ERI
eri = A.get_MO('aaaa')


# === 6. Use CCSD Final Energy as "exact" reference ===
exact_energy = A.FinalEnergy + rhf_e

# === 7. Optionally run Psi4’s built-in CCSD for comparison ===
psi4.set_options({'freeze_core': 'true'})
ccsd_e, ccsd_wfn = psi4.energy('ccsd', return_wfn=True)

# === 8. Extract Amplitudes ===
t1 = A.t1
t2 = A.t2

# === 9. Print Summary ===
print("CCSD correlation energy (Helper):", A.ccsd_corr_e)
print("Total CCSD energy (Helper):      ", A.FinalEnergy)
print("H1 shape:", hcore.shape)
print("ERI shape:", eri.shape)
print("Active electrons: (α, β) =", (num_elec_a, num_elec_b))
print("Exact (CCSD) Energy:", exact_energy)


Computing RHF reference.


/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/home/amousso3/DDLUCJ/check_amplitudes/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})


Cutting 2 core orbitals.

Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 10 basis functions.
(10, 10)
(8, 8)
Building initial guess...

..initialized CCSD in 0.039 seconds.

CCSD Iteration   0: CCSD correlation = -0.154646624232686   dE =  1.54647E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.147891140590808   dE =  6.75548E-03   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.151291559003040   dE = -3.40042E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.152315840580268   dE = -1.02428E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.153350720965912   dE = -1.03488E-03   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.153514987399626   dE = -1.64266E-04   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.153509819978286   dE =  5.16742E-06   DIIS = 5
CCSD Iteration   7: CCSD correlation = -0.153512855321502   dE = -3.03534E-06   DIIS = 6
CCSD Iteration   8: CCSD correlation = -0.153512629622595   dE =  

In [10]:
import pyscf
import pyscf.cc
import pyscf.mcscf

# Specify molecule properties
open_shell = False
spin_sq = 0

# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom='/home/amousso3/DDLUCJ/check_amplitudes/diatomics/NN.xyz',
    basis="STO-3G",
    symmetry="c1",
    spin=int(text.split('\n')[1].split()[1])-1
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
num_orbitals = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
num_elec_a = (n_electrons + mol.spin) // 2
num_elec_b = (n_electrons - mol.spin) // 2
cas = pyscf.mcscf.CASCI(scf, num_orbitals, (num_elec_a, num_elec_b))
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), num_orbitals)

# Compute exact energy
exact_energy = cas.run().e_tot
print("H1 shape:", hcore.shape)
print("ERI shape:", eri.shape)
print("Active electrons: (α, β) =", (num_elec_a, num_elec_b))
print("Exact (CCSD) Energy:", exact_energy)

converged SCF energy = -107.496598623389
CASCI E = -107.654041534620  E(CI) = -31.2154826569843  S^2 = 0.0000000
H1 shape: (8, 8)
ERI shape: (8, 8, 8, 8)
Active electrons: (α, β) = (5, 5)
Exact (CCSD) Energy: -107.65404153461981


In [ ]:
# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

In [ ]:
import ffsim
from qiskit import QuantumCircuit, QuantumRegister

n_reps = 1
alpha_alpha_indices = [(p, p + 1) for p in range(num_orbitals - 1)]
alpha_beta_indices = [(p, p) for p in range(0, num_orbitals, 4)]

ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(alpha_alpha_indices, alpha_beta_indices),
)

nelec = (num_elec_a, num_elec_b)

# create an empty quantum circuit
qubits = QuantumRegister(2 * num_orbitals, name="q")
circuit = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(num_orbitals, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()

backend = service.backend("ibm_sherbrooke")

In [ ]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

spin_a_layout = [0, 14, 18, 19, 20, 33, 39, 40, 41, 53, 60, 61, 62, 72, 81, 82]
spin_b_layout = [2, 3, 4, 15, 22, 23, 24, 34, 43, 44, 45, 54, 64, 65, 66, 73]
initial_layout = spin_a_layout + spin_b_layout

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, initial_layout=initial_layout
)

# without PRE_INIT passes
isa_circuit = pass_manager.run(circuit)
print(f"Gate counts (w/o pre-init passes): {isa_circuit.count_ops()}")

# with PRE_INIT passes
# We will use the circuit generated by this pass manager for hardware execution
pass_manager.pre_init = ffsim.qiskit.PRE_INIT
isa_circuit = pass_manager.run(circuit)
print(f"Gate counts (w/ pre-init passes): {isa_circuit.count_ops()}")

In [ ]:
from qiskit_ibm_runtime import SamplerV2 as Sampler

sampler = Sampler(mode=backend)
job = sampler.run([isa_circuit], shots=100_000)

In [ ]:
primitive_result = job.result()
pub_result = primitive_result[0]
counts = pub_result.data.meas.get_counts()

In [ ]:
from qiskit_addon_sqd.counts import counts_to_arrays

# Convert counts into bitstring and probability arrays
bitstring_matrix_full, probs_arr_full = counts_to_arrays(counts)

In [ ]:
import numpy as np
from qiskit_addon_sqd.configuration_recovery import recover_configurations
from qiskit_addon_sqd.fermion import solve_fermion
from qiskit_addon_sqd.subsampling import postselect_and_subsample

rng = np.random.default_rng(12345)

# SQD options
iterations = 5

# Eigenstate solver options
n_batches = 3
samples_per_batch = 1000
max_davidson_cycles = 200

# Self-consistent configuration recovery loop
e_hist = np.zeros((iterations, n_batches))  # energy history
s_hist = np.zeros((iterations, n_batches))  # spin history
occupancy_hist = []
avg_occupancy = None
for i in range(iterations):
    print(f"Starting configuration recovery iteration {i}")
    # On the first iteration, we have no orbital occupancy information from the
    # solver, so we just post-select from the full bitstring set based on hamming weight.
    if avg_occupancy is None:
        bs_mat_tmp = bitstring_matrix_full
        probs_arr_tmp = probs_arr_full

    # If we have average orbital occupancy information, we use it to refine the full set of noisy configurations
    else:
        bs_mat_tmp, probs_arr_tmp = recover_configurations(
            bitstring_matrix_full,
            probs_arr_full,
            avg_occupancy,
            num_elec_a,
            num_elec_b,
            rand_seed=rng,
        )

    # Throw out configurations with incorrect particle number in either the spin-up or spin-down systems
    batches = postselect_and_subsample(
        bs_mat_tmp,
        probs_arr_tmp,
        hamming_right=num_elec_a,
        hamming_left=num_elec_b,
        samples_per_batch=samples_per_batch,
        num_batches=n_batches,
        rand_seed=rng,
    )

    # Run eigenstate solvers in a loop. This loop should be parallelized for larger problems.
    e_tmp = np.zeros(n_batches)
    s_tmp = np.zeros(n_batches)
    occs_tmp = []
    coeffs = []
    for j in range(n_batches):
        energy_sci, coeffs_sci, avg_occs, spin = solve_fermion(
            batches[j],
            hcore,
            eri,
            open_shell=open_shell,
            spin_sq=spin_sq,
            max_davidson=max_davidson_cycles,
        )
        energy_sci += nuclear_repulsion_energy
        e_tmp[j] = energy_sci
        s_tmp[j] = spin
        occs_tmp.append(avg_occs)
        coeffs.append(coeffs_sci)

    # Combine batch results
    avg_occupancy = np.mean(occs_tmp, axis=0)

    # Track optimization history
    e_hist[i, :] = e_tmp
    s_hist[i, :] = s_tmp
    occupancy_hist.append(avg_occupancy)

In [ ]:
import matplotlib.pyplot as plt

# Data for energies plot
x1 = range(iterations)
e_diff = [abs(np.min(energies) - exact_energy) for energies in e_hist]
yt1 = [1.0, 1e-1, 1e-2, 1e-3, 1e-4]

# Chemical accuracy (+/- 1 milli-Hartree)
chem_accuracy = 0.001

# Data for avg spatial orbital occupancy
y2 = occupancy_hist[-1][0] + occupancy_hist[-1][1]
x2 = range(len(y2))

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot energies
axs[0].plot(x1, e_diff, label="energy error", marker="o")
axs[0].set_xticks(x1)
axs[0].set_xticklabels(x1)
axs[0].set_yticks(yt1)
axs[0].set_yticklabels(yt1)
axs[0].set_yscale("log")
axs[0].set_ylim(1e-4)
axs[0].axhline(y=chem_accuracy, color="#BF5700", linestyle="--", label="chemical accuracy")
axs[0].set_title("Approximated Ground State Energy Error vs SQD Iterations")
axs[0].set_xlabel("Iteration Index", fontdict={"fontsize": 12})
axs[0].set_ylabel("Energy Error (Ha)", fontdict={"fontsize": 12})
axs[0].legend()

# Plot orbital occupancy
axs[1].bar(x2, y2, width=0.8)
axs[1].set_xticks(x2)
axs[1].set_xticklabels(x2)
axs[1].set_title("Avg Occupancy per Spatial Orbital")
axs[1].set_xlabel("Orbital Index", fontdict={"fontsize": 12})
axs[1].set_ylabel("Avg Occupancy", fontdict={"fontsize": 12})

plt.tight_layout()
plt.show()